# IAN 620 · Complete LangChain Tutorial
## LangChain APIs, Parameters, Functions, Patterns, RAG, Tools, and Agents

This notebook is intentionally **LangChain-focused**. The goal is not just to run code, but to understand the LangChain objects, their parameters, what they return, and when you would choose one function or pattern over another.

### What makes this notebook different

For each major LangChain component, we will answer:

1. **What is it?**
2. **Why would I use it?**
3. **What are the important parameters?**
4. **What does the function return?**
5. **What are the most useful methods?**
6. **When should I use an alternative?**

### Main topics

- LangChain package structure
- chat models and local Hugging Face models
- messages
- prompt templates
- few-shot prompting
- Runnables and LCEL
- `invoke`, `batch`, and `stream`
- retries, configuration, fallbacks, and debugging
- output parsers
- `Document` objects
- document loaders
- text splitters
- embeddings
- Chroma vector stores
- retrievers
- RAG
- `@tool`
- retriever tools
- DuckDuckGo as a LangChain tool
- tool binding
- `create_agent`
- structured output
- conversation history
- optional LangSmith tracing

> **Core examples do not require a paid LLM API key.** The notebook downloads an open Hugging Face model and local embedding model.

## 0. LangChain in one picture

LangChain is an **application framework**, not a language model.

A useful mental model is:

**input → prompt → model → parser → application output**

For retrieval applications:

**documents → splitter → embeddings → vector store → retriever → prompt → model**

For agents:

**messages → model → tool call → tool result → model → final answer**

LangChain provides standard interfaces for these pieces so we do not have to write a new integration layer every time we change a model, vector store, loader, or tool.

### LangChain package map

Current LangChain is intentionally modular.

| Package | Purpose | Typical imports |
|---|---|---|
| `langchain` | High-level agent APIs and framework entry points | `create_agent` |
| `langchain-core` | Core abstractions shared by integrations | prompts, messages, Runnables, tools, `Document` |
| `langchain-community` | Community-maintained integrations | loaders, DuckDuckGo, utilities |
| `langchain-huggingface` | Hugging Face integrations | `ChatHuggingFace`, `HuggingFacePipeline`, `HuggingFaceEmbeddings` |
| `langchain-chroma` | Chroma vector-store integration | `Chroma` |
| `langchain-text-splitters` | Text-splitting utilities | `RecursiveCharacterTextSplitter` |

### Why separate packages?

It keeps the core framework small and lets provider integrations evolve independently.

## 1. Install the packages

### Why these packages?

| Package | Why we need it |
|---|---|
| `langchain` | High-level LangChain API |
| `langchain-community` | CSV loader and DuckDuckGo integration |
| `langchain-huggingface` | Local model and embedding integration |
| `langchain-chroma` | Chroma vector-store integration |
| `langchain-text-splitters` | Text splitting |
| `chromadb` | Chroma database engine |
| `sentence-transformers` | Local embedding model backend |
| `transformers` | Local Hugging Face language model backend |
| `accelerate` | Easier device placement for local models |
| `duckduckgo-search` | Public web-search backend |
| `pydantic` | Typed structured-output schemas |

In [ ]:
!pip -q install -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    "sentence-transformers>=5.2.0" \
    "transformers>=4.52.4" \
    accelerate \
    duckduckgo-search \
    pydantic

## 2. Imports and version check

Checking versions is useful because LangChain evolves quickly. If a tutorial from the web uses an old import path, comparing package versions can explain the mismatch.

In [ ]:
import os
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import torch

import langchain
import chromadb

print('LangChain:', langchain.__version__)
print('Chroma:', chromadb.__version__)
print('CUDA available:', torch.cuda.is_available())

# Part I — Models, Messages, and Prompts

## 3. Local models with `HuggingFacePipeline`

`HuggingFacePipeline` wraps a local Hugging Face Transformers pipeline in the LangChain Runnable interface.

### Why use it?

- No paid API is required.
- The model can run inside Colab or on your own computer.
- It behaves like other LangChain model components: `.invoke()`, `.batch()`, `.stream()` where supported, configuration, tracing, and composition.

### Important `from_model_id()` parameters

| Parameter | Meaning | Example / guidance |
|---|---|---|
| `model_id` | Hugging Face repository ID | `Qwen/Qwen3-0.6B` |
| `task` | Transformers pipeline task | Usually `text-generation` for causal LLMs |
| `model_kwargs` | Arguments passed when loading the model | `torch_dtype`, device-related options, revisions |
| `pipeline_kwargs` | Arguments used during generation | `max_new_tokens`, `temperature`, `do_sample`, `return_full_text` |
| `batch_size` | Number of inputs processed together when batching | Increase only if memory allows |

### `model_kwargs` vs. `pipeline_kwargs`

`model_kwargs` changes **how the model is loaded**. `pipeline_kwargs` changes **how generation runs**.

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

LOCAL_MODEL = (
    "Qwen/Qwen3-1.7B"
    if torch.cuda.is_available()
    else "Qwen/Qwen3-0.6B"
)

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=LOCAL_MODEL,
    task="text-generation",
    model_kwargs={
        "torch_dtype": "auto",
    },
    pipeline_kwargs={
        "max_new_tokens": 180,
        "do_sample": False,
        "return_full_text": False,
    },
)

print('Loaded:', LOCAL_MODEL)

### Common generation parameters

These are model-generation parameters rather than LangChain-only concepts, but LangChain passes them through to the underlying model.

| Parameter | What it changes | Typical effect |
|---|---|---|
| `max_new_tokens` | Maximum tokens generated after the prompt | Controls answer length/cost/memory |
| `do_sample=False` | Use deterministic token selection | Good for reproducible teaching examples |
| `do_sample=True` | Sample from the probability distribution | More varied output |
| `temperature` | Flattens/sharpens probabilities when sampling | Lower = more conservative; higher = more varied |
| `top_p` | Nucleus-sampling cutoff | Restricts sampling to a probability mass |
| `return_full_text=False` | Return only generated continuation | Avoid repeating the entire prompt in output |

**Important:** `temperature` usually matters only when sampling is enabled.

## 4. `ChatHuggingFace`: turn the local model into a chat model

`HuggingFacePipeline` is an LLM-style component. `ChatHuggingFace` adapts it to LangChain's **chat-model interface**.

### Why use the chat-model interface?

Chat models understand message roles such as system, human, AI, and tool messages. Most modern LangChain agent APIs expect a chat model.

In [ ]:
chat_model = ChatHuggingFace(llm=hf_llm)

response = chat_model.invoke(
    "Explain LangChain in two short sentences."
)

print(type(response).__name__)
print(response.content)

### What `.invoke()` returns

For a chat model, `.invoke()` returns an `AIMessage`, not a plain string.

Useful fields include:

| Field | Purpose |
|---|---|
| `.content` | Main generated content |
| `.tool_calls` | Structured tool requests when tool calling is used |
| `.response_metadata` | Provider/runtime metadata when available |
| `.usage_metadata` | Token usage when supported by the integration |

This distinction matters because the message object can carry more information than just text.

## 5. LangChain messages

Messages are the standard input/output objects for chat models.

| Message | Who produced it? | Typical use |
|---|---|---|
| `SystemMessage` | Application | Instructions, role, constraints |
| `HumanMessage` | User | User request |
| `AIMessage` | Model | Model response or tool request |
| `ToolMessage` | Tool runtime | Tool result returned to the model |

### Why not just concatenate strings?

Separate message roles preserve conversational structure and make tool calling and agent loops possible.

In [ ]:
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

messages = [
    SystemMessage(
        content="You are a concise Python instructor."
    ),
    HumanMessage(
        content="What is a Python dictionary?"
    ),
]

reply = chat_model.invoke(messages)
print(reply.content)

## 6. `PromptTemplate`

`PromptTemplate` is designed for **string prompts**.

### Why use it?

- Keep prompt wording separate from runtime values.
- Reuse the same prompt with many inputs.
- Validate required variables.
- Compose the prompt with models and parsers as a Runnable.

### Important parameters

| Parameter | Meaning |
|---|---|
| `template` | The string containing placeholders such as `{topic}` |
| `input_variables` | Variables that must be supplied at runtime |
| `partial_variables` | Variables filled in ahead of time |
| `template_format` | Formatting style; normally standard Python format strings |
| `validate_template` | Whether to validate variable consistency when supported |
| `input_types` | Optional expected variable types; strings are assumed otherwise |

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Explain {topic} to a beginner.
Give one simple example.
Keep the answer under {words} words."""
)

prompt_value = prompt.invoke({
    "topic": "vector embeddings",
    "words": 100,
})

print(prompt_value.text)

### Useful prompt methods

| Method | Use |
|---|---|
| `.invoke({...})` | Produce a prompt value using Runnable syntax |
| `.format(...)` | Produce a formatted string |
| `.partial(...)` | Pre-fill some variables and return a new template |
| `.input_variables` | Inspect required runtime variables |

`partial()` is useful for values that stay constant across many calls, such as course name, organization policy, or output language.

In [ ]:
base_prompt = PromptTemplate.from_template(
    "You teach {course}. Explain {topic} in {style} language."
)

nlp_prompt = base_prompt.partial(
    course="Natural Language Processing"
)

print(nlp_prompt.format(
    topic="tokenization",
    style="beginner-friendly",
))

## 7. `ChatPromptTemplate`

`ChatPromptTemplate` is the prompt class to use when the model expects message roles.

### `from_messages()`

`ChatPromptTemplate.from_messages(...)` accepts a sequence of message representations. The most common representation is a tuple:

`(role, template)`

Common roles include `system`, `human`, and `ai`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful instructor for {course}.",
    ),
    (
        "human",
        "Explain {topic} with one example.",
    ),
])

prompt_value = chat_prompt.invoke({
    "course": "NLP",
    "topic": "cosine similarity",
})

print(prompt_value)

### `MessagesPlaceholder`

`MessagesPlaceholder` reserves a position in the prompt for a **list of existing messages**.

This is useful for:

- conversation history,
- agent scratchpads/tool messages,
- injecting previously collected messages.

Important parameters:

| Parameter | Meaning |
|---|---|
| `variable_name` | Dictionary key containing the list of messages |
| `optional=True` | Allows the prompt to work even when no messages are supplied |

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

history_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful tutor.'),
    MessagesPlaceholder(
        variable_name='history',
        optional=True,
    ),
    ('human', '{question}'),
])

history_prompt.invoke({
    "history": [
        HumanMessage(content='I am learning LangChain.'),
        AIMessage(content='Great. We can build it step by step.'),
    ],
    "question": "What did I say I was learning?",
})

## 8. Few-shot prompting

Few-shot prompting gives the model examples of the behavior you want.

LangChain can keep examples separate from the instruction template, which makes examples easier to add, remove, or select dynamically.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

examples = [
    {
        "input": "happy",
        "output": "positive",
    },
    {
        "input": "terrible",
        "output": "negative",
    },
]

example_prompt = PromptTemplate.from_template(
    "Text: {input}\nLabel: {output}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Classify the sentiment.",
    suffix="Text: {text}\nLabel:",
    input_variables=['text'],
)

print(few_shot_prompt.format(text='excellent'))

### Key `FewShotPromptTemplate` parameters

| Parameter | Meaning |
|---|---|
| `examples` | Static list of example dictionaries |
| `example_prompt` | Template used to render each example |
| `prefix` | Text before the examples |
| `suffix` | Text after the examples; often includes the new input |
| `input_variables` | Runtime variables required by the final prompt |
| `example_separator` | String inserted between rendered examples |
| `example_selector` | Optional dynamic mechanism for choosing examples instead of passing all examples |

# Part II — Runnables and LCEL

## 9. The `Runnable` interface

`Runnable` is one of the most important LangChain abstractions.

A Runnable is a unit of work that can be:

- invoked once,
- batched,
- streamed,
- configured,
- retried,
- composed with other Runnables.

LangChain models, prompts, retrievers, parsers, and many chains implement this interface.

### Standard Runnable execution methods

| Method | What it does | Best use |
|---|---|---|
| `.invoke(input)` | One input → one output | Normal synchronous call |
| `.ainvoke(input)` | Async one input | Async applications |
| `.batch(inputs)` | Many inputs → many outputs | Throughput |
| `.abatch(inputs)` | Async batch | High-concurrency apps |
| `.stream(input)` | Yield output chunks | Progressive UI/output |
| `.astream(input)` | Async stream | Async streaming apps |

Because these methods are shared, learning them once applies to many LangChain components.

## 10. LCEL and the `|` operator

LangChain Expression Language (LCEL) lets us compose Runnables using the pipe operator.

`A | B | C` means:

1. run `A`,
2. pass its output to `B`,
3. pass `B`'s output to `C`.

This produces a `RunnableSequence`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in three sentences."
)

explain_chain = (
    explain_prompt
    | chat_model
    | StrOutputParser()
)

print(explain_chain.invoke({
    "topic": "TF-IDF"
}))

### Why `StrOutputParser()`?

A chat model returns an `AIMessage`. `StrOutputParser()` returns only the text content.

Without parser:

`ChatPromptValue → AIMessage`

With parser:

`ChatPromptValue → AIMessage → str`

## 11. `RunnableLambda`

`RunnableLambda` converts a normal Python callable into a LangChain Runnable.

### Why use it?

- Insert ordinary Python logic into an LCEL chain.
- Gain Runnable methods such as `.invoke()`, `.batch()`, tracing, and configuration.
- Keep small transformations readable.

`RunnableLambda` is best when the function does **not** need to emit incremental streaming chunks.

In [ ]:
from langchain_core.runnables import RunnableLambda


def clean_text(text: str) -> str:
    return ' '.join(text.strip().split())


cleaner = RunnableLambda(clean_text)

print(cleaner.invoke(
    "   LangChain    uses    Runnables.   "
))

## 12. `RunnableParallel`

`RunnableParallel` sends the **same input** to multiple Runnables and returns a dictionary of results.

Use it when independent operations can happen side by side.

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel({
    "original": RunnableLambda(lambda x: x),
    "upper": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(len),
})

parallel.invoke('LangChain')

### `RunnableParallel` vs. `RunnableSequence`

| Pattern | Behavior |
|---|---|
| Sequence | Output of one step becomes input to the next |
| Parallel | Same input is sent to several independent branches |

A Python dictionary inside an LCEL pipeline is automatically coerced into a `RunnableParallel`.

## 13. `RunnablePassthrough`

`RunnablePassthrough` returns the input unchanged.

It is especially useful in RAG because the original user question often needs to travel through the chain while another branch retrieves context.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()
print(passthrough.invoke('original input'))

### `.assign()` on `RunnablePassthrough`

`assign()` adds new keys to a dictionary while keeping existing keys.

This is useful when you want to enrich state-like dictionaries inside an LCEL pipeline.

In [ ]:
enrich = RunnablePassthrough.assign(
    length=lambda x: len(x['text']),
    upper=lambda x: x['text'].upper(),
)

enrich.invoke({
    "text": "LangChain"
})

## 14. Runnable configuration with `.with_config()`

`with_config()` attaches metadata used during execution and tracing.

Typical configuration values include:

| Config field | Purpose |
|---|---|
| `run_name` | Human-readable name for the step |
| `tags` | Labels used for filtering traces |
| `metadata` | Extra diagnostic information |

This becomes very useful when a chain has many steps.

In [ ]:
named_chain = explain_chain.with_config({
    "run_name": "beginner_explainer",
    "tags": ["tutorial", "langchain"],
    "metadata": {"course": "IAN 620"},
})

print(named_chain.invoke({
    "topic": "embeddings"
}))

## 15. Retry policies with `.with_retry()`

Many Runnables can be wrapped with a retry policy.

### Why use retries?

Retries are useful for **transient failures** such as temporary service errors. They should not be used to hide deterministic bugs.

Important parameters include:

| Parameter | Meaning |
|---|---|
| `stop_after_attempt` | Maximum number of attempts |
| `wait_exponential_jitter` | Whether to add exponential backoff/jitter between retries |

Apply retries to the smallest sensible step rather than retrying an entire expensive workflow.

In [ ]:
retrying_chain = explain_chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=False,
)

print(retrying_chain.invoke({
    "topic": "retrievers"
}))

## 16. Fallbacks with `.with_fallbacks()`

A fallback is a second Runnable used only if the primary Runnable fails.

Typical uses:

- primary model → backup model,
- primary service → backup service,
- complex parser → simpler parser.

Fallbacks are different from retries: a retry runs the same logic again; a fallback changes the implementation.

In [ ]:
primary = RunnableLambda(
    lambda x: x.upper()
)

backup = RunnableLambda(
    lambda x: f"Backup: {x}"
)

safe_runnable = primary.with_fallbacks([backup])

print(safe_runnable.invoke('langchain'))

## 17. Inspecting input and output schemas

Runnables expose schema information that can help debugging, documentation, and tool construction.

In [ ]:
print('Input schema:')
print(explain_chain.input_schema.model_json_schema())

print('\nOutput schema:')
print(explain_chain.output_schema.model_json_schema())

# Part III — Output Parsers

## 18. `StrOutputParser`

Use `StrOutputParser` when the next part of your application wants plain text instead of an `AIMessage`.

In [ ]:
string_chain = (
    ChatPromptTemplate.from_template(
        "Define {term} in one sentence."
    )
    | chat_model
    | StrOutputParser()
)

result = string_chain.invoke({
    "term": "retriever"
})

print(type(result).__name__)
print(result)

## 19. `JsonOutputParser`

`JsonOutputParser` asks the model for JSON-like structured output and parses it into Python data.

Use it when you need a dictionary/list but do not need a strongly typed Pydantic object.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = PromptTemplate.from_template(
    """Return a JSON object with keys "term" and "definition".
Term: {term}

{format_instructions}"""
).partial(
    format_instructions=json_parser.get_format_instructions()
)

json_chain = json_prompt | hf_llm | json_parser

try:
    print(json_chain.invoke({
        "term": "embedding"
    }))
except Exception as exc:
    print('Small local models may occasionally produce invalid JSON.')
    print(type(exc).__name__)

## 20. `PydanticOutputParser`

`PydanticOutputParser` parses model output into a typed Pydantic model.

### Why use it?

- Validate field names and data types.
- Make downstream Python code safer.
- Turn free-form model text into an application contract.

Main parameter:

`pydantic_object` — the Pydantic model class that defines the desired schema.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class ConceptCard(BaseModel):
    term: str = Field(description='Concept name')
    definition: str = Field(description='Short definition')
    example: str = Field(description='Short example')


card_parser = PydanticOutputParser(
    pydantic_object=ConceptCard
)

card_prompt = PromptTemplate.from_template(
    """Explain {topic}.
{format_instructions}"""
).partial(
    format_instructions=card_parser.get_format_instructions()
)

card_chain = card_prompt | hf_llm | card_parser

In [ ]:
try:
    card = card_chain.invoke({
        "topic": "cosine similarity"
    })
    print(card)
except Exception as exc:
    print('The local model did not produce valid structured output.')
    print(type(exc).__name__)

# Part IV — Documents and Loaders

## 21. `Document`

`Document` is LangChain's basic text container.

Two fields matter most:

| Field | Purpose |
|---|---|
| `page_content` | The actual text that may be split, embedded, or retrieved |
| `metadata` | Structured information carried with the text |

Metadata often stores:

- source path or URL,
- document ID,
- category,
- author,
- timestamp,
- permissions,
- section name.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Standard returns are accepted within 14 days of delivery.",
    metadata={
        "source_id": "KB001",
        "category": "returns",
    },
)

print(doc.page_content)
print(doc.metadata)

## 22. `CSVLoader`

`CSVLoader` converts each CSV row into a LangChain `Document`.

### Important parameters

| Parameter | Meaning |
|---|---|
| `file_path` | Path to the CSV file |
| `source_column` | Column whose value becomes document source metadata |
| `metadata_columns` | Columns copied into metadata |
| `content_columns` | Columns used to build `page_content` |
| `csv_args` | Arguments passed to Python's CSV reader, such as delimiter |
| `encoding` | File encoding |
| `autodetect_encoding` | Try to detect encoding automatically |

### Useful methods

| Method | Meaning |
|---|---|
| `.load()` | Load all documents into memory |
| `.lazy_load()` | Yield documents one at a time |
| `.aload()` | Async load |
| `.alazy_load()` | Async lazy loading |

`lazy_load()` is useful for very large files because it avoids creating the entire list at once.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base.csv"
CSV_FILE = "langchain_tutorial_kb.csv"

urllib.request.urlretrieve(
    DATA_URL,
    CSV_FILE,
)

print('Downloaded:', CSV_FILE)

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path=CSV_FILE,
    source_column='source_id',
    metadata_columns=[
        'source_id',
        'category',
        'effective_date',
        'source_url',
    ],
    content_columns=[
        'title',
        'text',
    ],
)

loaded_docs = loader.load()

print('Loaded documents:', len(loaded_docs))
print(loaded_docs[0])

### When to construct `Document` objects manually

Loaders are convenient, but manual construction gives maximum control over:

- exactly which fields become content,
- exactly which fields become metadata,
- custom IDs,
- preprocessing rules.

Use loaders for standard file formats; use manual construction when domain-specific transformations matter.

# Part V — Text Splitting

## 23. `RecursiveCharacterTextSplitter`

This is a common general-purpose splitter.

Instead of cutting every N characters blindly, it tries a sequence of separators to preserve larger logical units where possible.

### Important parameters

| Parameter | Meaning | Guidance |
|---|---|---|
| `chunk_size` | Maximum target chunk length | Larger preserves context; smaller increases retrieval precision |
| `chunk_overlap` | Repeated content between adjacent chunks | Helps preserve facts crossing boundaries |
| `length_function` | Function used to measure length | Default is character count |
| `separators` | Ordered separators the splitter tries | Useful for domain-specific structure |
| `is_separator_regex` | Treat separators as regular expressions | Use only when needed |

### Choosing values

There is no universally correct chunk size. Evaluate retrieval on the questions your application actually receives.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
)

chunks = splitter.split_documents(
    loaded_docs
)

print('Documents:', len(loaded_docs))
print('Chunks:', len(chunks))

### `.split_text()` vs. `.split_documents()`

| Method | Input | Output |
|---|---|---|
| `.split_text(text)` | One string | `list[str]` |
| `.split_documents(docs)` | `list[Document]` | `list[Document]` with metadata preserved |

For RAG, `split_documents()` is usually preferable because metadata remains attached to each chunk.

# Part VI — Embeddings

## 24. `HuggingFaceEmbeddings`

`HuggingFaceEmbeddings` integrates Sentence Transformers embedding models with LangChain.

### Important parameters

| Parameter | Meaning |
|---|---|
| `model_name` | Sentence Transformers / Hugging Face embedding model ID |
| `model_kwargs` | Arguments passed when creating the Sentence Transformer |
| `encode_kwargs` | Arguments passed during `.encode()` |
| `cache_folder` | Optional model-cache location |
| `multi_process` | Whether to use multi-process encoding when supported |

### Useful `model_kwargs` examples

- `device='cuda'` — force GPU.
- `device='cpu'` — force CPU.
- `revision='...'` — pin a model revision.
- `trust_remote_code=True` — only when you trust the model repository and it requires custom code.

### Useful `encode_kwargs` examples

- `normalize_embeddings=True` — normalize vectors.
- `batch_size=32` — embedding batch size.
- `precision='float32'` or supported alternatives — encoding precision.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    },
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

### `embed_query()` vs. `embed_documents()`

| Method | Input | Output |
|---|---|---|
| `.embed_query(text)` | One query string | One vector |
| `.embed_documents(texts)` | List of strings | List of vectors |

Vector stores call these methods internally, but testing them directly is useful for understanding the embedding interface.

In [ ]:
query_vector = embeddings.embed_query(
    "return policy after delivery"
)

document_vectors = embeddings.embed_documents([
    "Standard return window is 14 days.",
    "Refunds take several business days.",
])

print('Query dimensions:', len(query_vector))
print('Document vectors:', len(document_vectors))

# Part VII — Vector Stores and Chroma

## 25. `Chroma.from_documents()`

`Chroma.from_documents()` is a convenient constructor that:

1. accepts LangChain `Document` objects,
2. embeds their text,
3. stores vectors plus metadata,
4. returns a LangChain Chroma vector-store object.

### Important parameters

| Parameter | Meaning |
|---|---|
| `documents` | List of `Document` objects to index |
| `embedding` | LangChain embedding object |
| `ids` | Optional stable IDs for documents/chunks |
| `collection_name` | Name of the Chroma collection |
| `persist_directory` | Local directory for persistent storage |
| `collection_metadata` | Optional collection-level metadata |
| `client` / `client_settings` | Advanced control over the Chroma client |

If `persist_directory` is omitted, the collection may be ephemeral depending on the client configuration.

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = "./langchain_tutorial_chroma"
COLLECTION = "langchain_tutorial"

shutil.rmtree(
    CHROMA_DIR,
    ignore_errors=True,
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[
        f'chunk-{i:03d}'
        for i in range(len(chunks))
    ],
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
)

print('Stored IDs:', len(vector_store.get()['ids']))

## 26. Chroma search functions

LangChain vector stores expose several useful search styles.

| Function | What it returns | When to use |
|---|---|---|
| `.similarity_search(query, k=...)` | Documents | Normal semantic retrieval |
| `.similarity_search_with_score(...)` | `(Document, score)` pairs | Debugging / inspecting distances |
| `.max_marginal_relevance_search(...)` | Diverse relevant documents | Reduce near-duplicate results |

### `k`

`k` controls the number of final documents returned. Larger `k` gives more context but may introduce noise.

In [ ]:
query = "How long do I have to return a normal item?"

results = vector_store.similarity_search(
    query,
    k=3,
)

for d in results:
    print(
        d.metadata.get('source_id'),
        '→',
        d.page_content[:160],
    )

In [ ]:
scored_results = vector_store.similarity_search_with_score(
    query,
    k=3,
)

for d, score in scored_results:
    print(
        d.metadata.get('source_id'),
        'score =',
        round(float(score), 4),
    )

### Metadata filtering

A filter restricts retrieval to documents whose metadata matches specified conditions.

Use filters when the application already knows something important, for example:

- user has permission to see only one department,
- search should use only current policy documents,
- question belongs to one category.

In [ ]:
refund_docs = vector_store.similarity_search(
    "How long does a refund take?",
    k=3,
    filter={
        "category": "refunds"
    },
)

[(d.metadata.get('source_id'), d.metadata.get('category')) for d in refund_docs]

## 27. MMR search

**Maximum Marginal Relevance (MMR)** balances relevance to the query with diversity among selected documents.

Important parameters:

| Parameter | Meaning |
|---|---|
| `k` | Final number of documents returned |
| `fetch_k` | Number of initial similarity candidates considered |
| `lambda_mult` | Similarity vs. diversity tradeoff |

For `lambda_mult`, values closer to **1** emphasize similarity; values closer to **0** emphasize diversity.

In [ ]:
mmr_docs = vector_store.max_marginal_relevance_search(
    "returns and refund rules",
    k=3,
    fetch_k=8,
    lambda_mult=0.5,
)

[(d.metadata.get('source_id'), d.metadata.get('category')) for d in mmr_docs]

# Part VIII — Retrievers

## 28. `.as_retriever()`

A vector store performs search. A **retriever** exposes retrieval through LangChain's standard retriever interface.

### Why convert to a retriever?

Your RAG chain can depend on `retriever.invoke(question)` rather than directly depending on Chroma. This makes the rest of the application easier to swap to another retriever later.

### Important parameters

| Parameter | Meaning |
|---|---|
| `search_type` | Retrieval strategy |
| `search_kwargs` | Parameters passed to that search strategy |

Common `search_type` values for vector stores include:

- `similarity`
- `mmr`
- `similarity_score_threshold`

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    },
)

docs = retriever.invoke(
    "What is the warranty period?"
)

[(d.metadata.get('source_id'), d.metadata.get('title')) for d in docs]

### MMR retriever parameters

Example:

```python
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    },
)
```

`fetch_k` creates the initial candidate set. MMR then chooses `k` documents from those candidates.

In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 8,
        "lambda_mult": 0.5,
    },
)

mmr_retriever.invoke(
    "returns refunds damaged items"
)

### Similarity score threshold

This mode can return fewer than `k` results if documents do not meet the threshold.

Use it when **low-quality retrieval should produce no evidence rather than weak evidence**.

In [ ]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.25,
    },
)

threshold_retriever.invoke(
    "How can I reset my account password?"
)

# Part IX — RAG with LangChain

## 29. Manual RAG first

Before composing RAG with LCEL, it is useful to see the individual steps.

1. Call the retriever.
2. Format the returned `Document` objects.
3. Fill the prompt.
4. Call the model.
5. Parse the output.

In [ ]:
def format_docs(docs):
    return '\n\n'.join(
        f"[{d.metadata.get('source_id', 'source')}] {d.page_content}"
        for d in docs
    )


rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the context is insufficient, say that you cannot verify the answer.
Cite source IDs such as [KB001].

CONTEXT:
{context}

QUESTION:
{question}"""
)

In [ ]:
question = 'Can I return a standard item after 20 days?'

evidence = retriever.invoke(question)
context = format_docs(evidence)

prompt_value = rag_prompt.invoke({
    "context": context,
    "question": question,
})

model_response = chat_model.invoke(prompt_value)

print(model_response.content)

## 30. RAG with LCEL

Now the same steps become one composable Runnable.

The dictionary below is interpreted as a parallel Runnable:

- `context` branch → retriever → formatting function
- `question` branch → pass the original question through unchanged

In [ ]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

print(rag_chain.invoke(
    "How quickly should shipping damage be reported?"
))

### Why this LCEL RAG pattern is useful

Each component can be replaced independently:

- Chroma retriever → another retriever
- prompt → another prompt
- local model → provider model
- string parser → structured parser

The outer application can still call `.invoke()` in the same way.

# Part X — Tools

## 31. The `@tool` decorator

`@tool` converts a Python function into a LangChain tool.

LangChain infers a tool schema from:

- the function name,
- the docstring,
- type hints,
- optional explicit schema configuration.

### Why do type hints matter?

The tool schema tells a model what arguments are valid. Good type hints produce better schemas.

### Important ideas

| Concept | Why it matters |
|---|---|
| Function name | Becomes the default tool name |
| Docstring | Tells the model when to use the tool |
| Type hints | Define argument types |
| Return value | Becomes tool output |
| `args_schema` | Optional explicit Pydantic argument schema |

Avoid tool argument names such as `config`, `run_manager`, or `callbacks` because LangChain reserves them for runtime behavior.

In [ ]:
from langchain.tools import tool


@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


print('Name:', add_numbers.name)
print('Description:', add_numbers.description)
print('Args schema:', add_numbers.args)

In [ ]:
print(add_numbers.invoke({
    "a": 12.5,
    "b": 7.5,
}))

## 32. Explicit tool argument schemas

For more control, define a Pydantic model and pass it as `args_schema`.

This is useful for:

- field descriptions,
- validation,
- enums or constraints,
- more readable tool schemas.

In [ ]:
from pydantic import BaseModel, Field


class DiscountInput(BaseModel):
    price: float = Field(
        gt=0,
        description='Original price',
    )
    percent: float = Field(
        ge=0,
        le=100,
        description='Discount percentage from 0 to 100',
    )


@tool(args_schema=DiscountInput)
def calculate_discount(
    price: float,
    percent: float,
) -> float:
    """Calculate the price after a percentage discount."""
    return price * (1 - percent / 100)


calculate_discount.invoke({
    "price": 100,
    "percent": 15,
})

## 33. `create_retriever_tool()`

LangChain provides a helper specifically for turning a retriever into a tool.

### Parameters

| Parameter | Meaning |
|---|---|
| `retriever` | The `BaseRetriever` to call |
| `name` | Tool name visible to the model |
| `description` | Tool description visible to the model |
| `document_prompt` | Optional prompt used to render each retrieved document |
| `document_separator` | Separator inserted between rendered documents |
| `response_format` | `content` or `content_and_artifact` |

`content_and_artifact` is useful when the model should receive formatted text while the application also keeps the original retrieved documents as an artifact.

In [ ]:
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="search_support_knowledge",
    description=(
        'Search approved internal support policies. '
        'Use this for returns, refunds, shipping, warranties, accounts, '
        'billing, subscriptions, privacy, and related support questions.'
    ),
    document_separator="\n\n---\n\n",
)

print(retriever_tool.invoke({
    "query": "standard return window"
}))

## 34. DuckDuckGo as a LangChain tool

`DuckDuckGoSearchRun` is an integration that already implements the tool-like interface.

Use it for public web information. Do not use web search to replace authoritative internal policy.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

web_search = DuckDuckGoSearchRun()

try:
    result = web_search.invoke(
        "LangChain Python official documentation"
    )
    print(str(result)[:1200])
except Exception as exc:
    print('Web search unavailable in this runtime.')
    print(type(exc).__name__)

# Part XI — Tool Calling and Agents

## 35. `.bind_tools()`

`bind_tools(tools)` creates a new chat-model Runnable that knows the schemas of the supplied tools.

### What it does not do

It does **not** execute the tools automatically by itself.

The model may return an `AIMessage` containing `.tool_calls`. Your application—or an agent harness—must execute those calls and return the results as `ToolMessage` objects.

In [ ]:
tools = [
    add_numbers,
    retriever_tool,
    web_search,
]

try:
    model_with_tools = chat_model.bind_tools(
        tools
    )

    tool_decision = model_with_tools.invoke(
        "Use the internal policy tool to find the standard return window."
    )

    print('Content:', tool_decision.content)
    print('Tool calls:', tool_decision.tool_calls)
except Exception as exc:
    model_with_tools = None
    print('This small local model/runtime did not complete tool binding.')
    print(type(exc).__name__)

## 36. Manual tool loop

This is the logic that agent frameworks automate:

1. model receives messages,
2. model requests a tool,
3. application executes the tool,
4. application creates a `ToolMessage`,
5. model receives the tool result,
6. repeat until there are no tool calls.

In [ ]:
if model_with_tools is not None:
    conversation = [
        HumanMessage(
            content="Use the calculator to add 18.5 and 7.25."
        )
    ]

    ai_msg = model_with_tools.invoke(
        conversation
    )
    conversation.append(ai_msg)

    tool_map = {
        t.name: t
        for t in tools
    }

    for call in ai_msg.tool_calls:
        selected_tool = tool_map[call['name']]
        result = selected_tool.invoke(call['args'])

        conversation.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call['id'],
                name=call['name'],
            )
        )

    if ai_msg.tool_calls:
        final_msg = model_with_tools.invoke(
            conversation
        )
        print(final_msg.content)
    else:
        print(ai_msg.content)
else:
    print('Manual tool loop skipped.')

## 37. `create_agent()`

`create_agent()` is LangChain's current high-level agent constructor.

It creates an agent graph that repeatedly calls the model and tools until a stopping condition is reached.

### Important parameters

| Parameter | Purpose |
|---|---|
| `model` | Chat model or supported model identifier |
| `tools` | Tools available to the agent |
| `system_prompt` | High-level instructions |
| `middleware` | Hooks that can modify model/tool behavior |
| `response_format` | Optional structured-output schema/strategy |
| `checkpointer` | Optional persistence for thread/conversation state |
| `store` | Optional cross-thread store |
| `context_schema` | Optional runtime context schema |
| `name` | Optional graph/agent name |

### When to use `create_agent()`

Use it when the standard **model ↔ tool loop** is enough.

If you need highly custom control flow, branching, complex state, or explicit multi-stage workflows, use a lower-level orchestration approach.

In [ ]:
from langchain.agents import create_agent

agent = None

try:
    agent = create_agent(
        model=chat_model,
        tools=tools,
        system_prompt=(
            'You are a careful assistant. '
            'Use search_support_knowledge for internal support policy. '
            'Use DuckDuckGo only for explicitly public/current information. '
            'Use add_numbers for arithmetic.'
        ),
    )

    print('Agent created.')
except Exception as exc:
    print('This small local model could not initialize the agent reliably.')
    print(type(exc).__name__)

### Agent input format

A common invocation shape is:

```python
agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Question here",
        }
    ]
})
```

The result contains updated agent state, including its message history.

In [ ]:
if agent is not None:
    try:
        result = agent.invoke({
            'messages': [
                {
                    "role": "user",
                    "content": "What is the standard return window in our internal policy?",
                }
            ]
        })

        print(result['messages'][-1].content)
    except Exception as exc:
        print('The small local model did not complete this tool-using run.')
        print(type(exc).__name__)

## 38. Structured agent output

`create_agent()` can be given a response schema using `response_format`.

LangChain can choose an appropriate structured-output strategy based on model capabilities.

This is useful when downstream application code expects typed fields rather than prose.

In [ ]:
class SupportDecision(BaseModel):
    category: str
    answer: str
    needs_human: bool


try:
    structured_agent = create_agent(
        model=chat_model,
        tools=[retriever_tool],
        response_format=SupportDecision,
        system_prompt=(
            'Use internal policy evidence when needed. '
            'Return a structured support decision.'
        ),
    )

    structured_result = structured_agent.invoke({
        'messages': [
            {
                "role": "user",
                "content": "Can a standard item be returned 20 days after delivery?",
            }
        ]
    })

    print(
        structured_result.get(
            "structured_response"
        )
    )
except Exception as exc:
    print('This small local model may not support reliable structured-agent output.')
    print(type(exc).__name__)

# Part XII — Conversation History

## 39. `RunnableWithMessageHistory`

`RunnableWithMessageHistory` wraps another Runnable and automatically loads/saves chat history for a session.

### Important parameters

| Parameter | Meaning |
|---|---|
| `runnable` | Chain being wrapped |
| `get_session_history` | Function that returns a history object for a session ID |
| `input_messages_key` | Key containing the newest user input |
| `history_messages_key` | Key where previous messages should be inserted |

### Important distinction

Chat history is usually **short-term conversational state**. It is not automatically long-term semantic memory.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

conversation_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise assistant.'),
    MessagesPlaceholder(
        variable_name='history',
        optional=True,
    ),
    ('human', '{question}'),
])

conversation_chain = (
    conversation_prompt
    | chat_model
    | StrOutputParser()
)

history_store = {}


def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = (
            InMemoryChatMessageHistory()
        )
    return history_store[session_id]


chat_with_history = RunnableWithMessageHistory(
    conversation_chain,
    get_history,
    input_messages_key='question',
    history_messages_key='history',
)

In [ ]:
config = {"configurable": {"session_id": "demo-user"}}

print(chat_with_history.invoke(
    {"question": "I am learning LangChain."},
    config=config,
))

print(chat_with_history.invoke(
    {"question": "What did I say I was learning?"},
    config=config,
))

# Part XIII — Debugging and Observability

## 40. Built-in debug output

For local debugging, LangChain can print detailed execution information.

Use this temporarily because it can generate a lot of output.

In [ ]:
from langchain_core.globals import set_debug

# Turn on only when you want verbose LangChain debug output.
set_debug(False)

## 41. Per-run callbacks

Instead of enabling global debug output, callbacks can be passed through the Runnable configuration for a specific call.

Example:

```python
from langchain_core.tracers import ConsoleCallbackHandler

chain.invoke(
    input_value,
    config={"callbacks": [ConsoleCallbackHandler()]},
)
```

This is often cleaner than enabling debugging globally.

## 42. LangSmith tracing — optional

LangSmith is the LangChain ecosystem's tracing and evaluation platform.

It is useful when you need to inspect:

- prompts sent to models,
- retriever calls,
- tool calls,
- token usage,
- latency,
- agent trajectories,
- failures across many runs.

Typical environment setup:

```python
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = 'your-key'
```

Never commit secrets into a public notebook.

# Part XIV — Advanced LangChain Patterns

## 43. `.bind()`

`.bind()` creates a new Runnable with some call-time keyword arguments pre-configured.

Use it when the same parameter should be supplied repeatedly.

Example concept:

```python
model_with_stop = model.bind(stop=['END'])
```

The exact keyword arguments available depend on the underlying model integration.

## 44. Branching with `RunnableBranch`

`RunnableBranch` chooses a Runnable based on conditions.

This is useful for simple deterministic routing without building an agent.

In [ ]:
from langchain_core.runnables import RunnableBranch

router = RunnableBranch(
    (
        lambda x: 'sql' in x.lower(),
        RunnableLambda(
            lambda x: f"Database path: {x}"
        ),
    ),
    (
        lambda x: 'web' in x.lower(),
        RunnableLambda(
            lambda x: f"Web path: {x}"
        ),
    ),
    RunnableLambda(
        lambda x: f"Default path: {x}"
    ),
)

print(router.invoke(
    "Use SQL for this question"
))

### `RunnableBranch` vs. an agent

Use `RunnableBranch` when routing rules are deterministic and easy to express in code.

Use an agent when flexible model judgment is genuinely useful.

This distinction is important: not every conditional workflow needs an agent.

## 45. Batch configuration

`.batch()` can accept a `config` argument.

For integrations that perform concurrent work, `max_concurrency` can limit how much runs at once.

Example pattern:

```python
outputs = chain.batch(
    inputs,
    config={"max_concurrency": 4},
)
```

Use concurrency limits to respect API rate limits or local hardware capacity.

# Part XV — LangChain Integration Patterns

## 46. Swapping components without rewriting the whole app

One of LangChain's main benefits is interface stability.

For example, a RAG application can keep this outer structure:

```text
retriever | formatting | prompt | chat_model | parser
```

while independently replacing:

- local embeddings with hosted embeddings,
- Chroma with another vector store,
- the local chat model with a provider model,
- one document loader with another.

That is why understanding **interfaces** is more valuable than memorizing a specific provider.

## 47. Optional: Ollama as another LangChain chat-model integration

If you run models locally through Ollama, install:

```bash
pip install -U langchain-ollama
ollama pull qwen3:4b
```

Then:

```python
from langchain_ollama import ChatOllama

ollama_model = ChatOllama(
    model="qwen3:4b",
    temperature=0,
)

response = ollama_model.invoke(
    "Explain LangChain Runnables."
)
```

The rest of the LangChain prompt/Runnable/tool code can remain largely the same.

## 48. Optional: provider-agnostic initialization

Current LangChain also provides `init_chat_model()` as a unified entry point for supported chat-model providers.

Conceptual pattern:

```python
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "provider:model-name"
)
```

This is useful when you want application code to depend on LangChain's model interface rather than a provider-specific class.

# Part XVI — Retrieval Evaluation

## 49. Evaluate retrieval separately from generation

A RAG answer can fail because:

1. the retriever returned the wrong evidence,
2. the prompt was poor,
3. the model misused good evidence.

Testing retrieval first tells us whether the evidence layer is working.

In [ ]:
tests = [
    ('What is the normal return deadline?', 'returns'),
    ('How long does refund processing take?', 'refunds'),
    ('When should shipping damage be reported?', 'damaged_items'),
    ('How long is the limited warranty?', 'warranty'),
    ('How can I reset my password?', 'accounts'),
]

rows = []

for question, expected_category in tests:
    docs = retriever.invoke(question)
    categories = [
        d.metadata.get('category', '')
        for d in docs
    ]

    rows.append({
        'question': question,
        'expected_category': expected_category,
        'retrieved_categories': ', '.join(categories),
        'hit': expected_category in categories,
    })

evaluation = pd.DataFrame(rows)
evaluation

In [ ]:
print(
    'Retrieval hit rate:',
    f"{evaluation['hit'].mean():.0%}"
)

# Part XVII — LangChain API Cheat Sheet

## 50. Core objects

| Object | Main purpose | Main method |
|---|---|---|
| `ChatHuggingFace` | Chat-model interface | `.invoke()` |
| `PromptTemplate` | String prompt template | `.format()` / `.invoke()` |
| `ChatPromptTemplate` | Message-based prompt | `.invoke()` |
| `MessagesPlaceholder` | Inject message history | used inside chat prompts |
| `RunnableLambda` | Wrap Python function | `.invoke()` |
| `RunnableParallel` | Parallel branches | `.invoke()` |
| `RunnablePassthrough` | Preserve input | `.invoke()` / `.assign()` |
| `RunnableBranch` | Conditional routing | `.invoke()` |
| `StrOutputParser` | AI message → string | `.invoke()` |
| `PydanticOutputParser` | Text → typed object | `.invoke()` |
| `Document` | Text + metadata | data object |
| `CSVLoader` | CSV → Documents | `.load()` |
| `RecursiveCharacterTextSplitter` | Documents → chunks | `.split_documents()` |
| `HuggingFaceEmbeddings` | Text → vectors | `.embed_query()` |
| `Chroma` | Store/search vectors | `.similarity_search()` |
| Retriever | Query → Documents | `.invoke()` |
| `@tool` | Python function → tool | `.invoke()` |
| `create_retriever_tool` | Retriever → tool | `.invoke()` |
| `create_agent` | Model/tool loop | `.invoke()` |

## 51. Runnable methods to remember

| Method | Why use it |
|---|---|
| `.invoke()` | One request |
| `.batch()` | Many requests |
| `.stream()` | Progressive output |
| `.with_config()` | Add run name/tags/metadata |
| `.with_retry()` | Retry transient failures |
| `.with_fallbacks()` | Switch to alternate Runnable on failure |
| `.bind()` | Preconfigure call-time kwargs |
| `.input_schema` | Inspect expected input |
| `.output_schema` | Inspect output type/schema |

## 52. Common mistakes

### Using old tutorials without checking the current API
Older LangChain examples often use classes or imports that have moved into `langchain-classic` or separate integration packages.

### Treating every object as a plain Python function
Many LangChain objects are Runnables. Use `.invoke()` rather than assuming normal function-call syntax.

### Ignoring return types
A chat model returns an `AIMessage`; a retriever returns `Document` objects; a parser may return a string, dict, or Pydantic model.

### Putting everything in one chain before testing components
Test the loader, splitter, embedding model, vector store, retriever, and tool separately first.

### Choosing retrieval parameters without evaluation
`k`, `fetch_k`, `lambda_mult`, chunk size, and filters should be evaluated against representative questions.

### Overusing agents
If the control flow is deterministic, normal Python or `RunnableBranch` is often simpler.

## 53. Suggested LangChain learning order

1. `.invoke()` and chat models
2. message objects
3. `PromptTemplate` and `ChatPromptTemplate`
4. output parsers
5. LCEL `|`
6. `RunnableLambda`, `RunnableParallel`, `RunnablePassthrough`
7. batching, streaming, retries, configuration
8. `Document` and loaders
9. splitters
10. embeddings
11. vector stores
12. retrievers
13. RAG
14. `@tool`
15. `bind_tools()`
16. manual tool loop
17. `create_agent()`
18. structured output and history
19. tracing and evaluation

## 54. Where LangGraph fits — only after LangChain

This notebook deliberately focuses on LangChain.

LangGraph becomes useful when the **control flow itself** needs to be explicitly modeled as persistent state, nodes, edges, loops, interrupts, or long-running execution.

For most learners, the best sequence is:

**LangChain components first → LangChain RAG/tools/agents → LangGraph only when custom orchestration becomes necessary.**

## 55. Official references

- LangChain overview: https://docs.langchain.com/oss/python/langchain/overview
- LangChain agents: https://docs.langchain.com/oss/python/langchain/agents
- LangChain tools: https://docs.langchain.com/oss/python/langchain/tools
- LangChain reference: https://reference.langchain.com/python/langchain
- Runnable reference: https://reference.langchain.com/python/langchain-core/runnables
- Chroma integration: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- Hugging Face pipelines: https://docs.langchain.com/oss/python/integrations/llms/huggingface_pipelines
- ChatHuggingFace: https://docs.langchain.com/oss/python/integrations/chat/huggingface

Use the conceptual documentation to learn patterns and the reference documentation to check exact parameters and method signatures.